In [1]:
import pandas as pd
import unicodedata

In [8]:
# ── FUNCION LIMPIEZA ───────────────────────────────────
def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    
    texto = str(texto)
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = texto.upper()
    texto = " ".join(texto.split())
    
    return texto

# ── FUNCION HOMOLOGACION GENERO ────────────────────────
def homologar_genero(g):
    if pd.isnull(g):
        return g
    
    g = limpiar_texto(g)
    
    if g in ["HOMBRE", "MASCULINO"]:
        return "HOMBRE"
    elif g in ["MUJER", "FEMENINO"]:
        return "MUJER"
    else:
        return "OTRO"

# ── FUNCION CATEGORIZACIÓN COMPLETA ────────────────────
def clasificar_categoria(nombre):
    if pd.isnull(nombre):
        return "PREGRADOS"
    
    # ya viene limpio, pero aseguramos
    nombre = limpiar_texto(nombre)
    
    if nombre.startswith("DOCTORADO"):
        return "DOCTORADOS"
    
    elif nombre.startswith((
        "ESPECIALIDAD", "ESPECIALIZACIN", "ESPECIALIZACION",
        "ESPECIALIZACION:", "ESPECIALIZALICION", "ESPECILIZACION"
    )):
        return "ESPECIALIZACIONES"
    
    elif nombre.startswith((
        "MAESTRA", "MAESTRAS", "MAESTRIA", "MAESTRIAS"
    )):
        return "MAESTRIAS"
    
    elif nombre.startswith((
        "TECNICA", "TECNICO", "TCNICA", "TCNICO",
        "TECNOLOGIA", "TECNOLOGO", "TECNOLGIA",
        "TECNOLOGA", "TECNOLLOGIA", "TECNONOLOGIA", "TEGNOLOGIA"
    )):
        return "TECNICOS Y TECNOLOGIAS"
    
    else:
        return "PREGRADOS"

In [ ]:
# ── RUTAS ──────────────────────────────────────────────
RUTA = "/home/jovyan/work"

ARCHIVO_PRED = RUTA + "/data_predicciones.csv"
ARCHIVO_DATA = RUTA + "/data.csv"

SALIDA = RUTA + "/data_predicciones_enriquecido.csv"

# ── CARGA ──────────────────────────────────────────────
df_pred = pd.read_csv(ARCHIVO_PRED)
df_data = pd.read_csv(ARCHIVO_DATA)

In [11]:
# ── LIMPIAR TEXTO BASE ─────────────────────────────────
df_data["genero"] = df_data["genero"].apply(limpiar_texto)

# ── HOMOLOGAR GENERO ───────────────────────────────────
df_data["genero"] = df_data["genero"].apply(homologar_genero)

# ── CREAR CATÁLOGOS LIMPIOS (SIN DUPLICADOS) ───────────
inst = (
    df_data[["codigo_institucion", "institucion"]]
    .drop_duplicates(subset=["codigo_institucion"])
)

prog = (
    df_data[["codigo_programa", "programa"]]
    .drop_duplicates(subset=["codigo_programa"])
)

mun = (
    df_data[["codigo_municipio", "municipio"]]
    .drop_duplicates(subset=["codigo_municipio"])
)

gen = (
    df_data[["codigo_genero", "genero"]]
    .drop_duplicates(subset=["codigo_genero"])
)

# ── LIMPIAR TEXTO EN TODOS ─────────────────────────────
for df in [inst, prog, mun]:
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(limpiar_texto)

# ── VALIDAR DUPLICADOS (DEBUG OPCIONAL) ────────────────
print("Duplicados genero:", gen["codigo_genero"].duplicated().sum())

Duplicados genero: 0


In [ ]:
# ── MERGES ─────────────────────────────────────
df_final = df_pred.merge(inst, on="codigo_institucion", how="left", validate="many_to_one")
df_final = df_final.merge(prog, on="codigo_programa", how="left", validate="many_to_one")
df_final = df_final.merge(mun, on="codigo_municipio", how="left", validate="many_to_one")
df_final = df_final.merge(gen, on="codigo_genero", how="left", validate="many_to_one")

In [13]:
# ── APLICAR CATEGORIA ─────────────────────────────────
df_final["categoria"] = df_final["programa"].apply(clasificar_categoria)

# ── LIMPIAR NOMBRES COLUMNAS ───────────────────────────
df_final.columns = [col.upper() for col in df_final.columns]

# ── VALIDACION FINAL ───────────────────────────────────
print("Filas predicciones:", len(df_pred))
print("Filas final:", len(df_final))

Filas predicciones: 311470
Filas final: 311470


In [15]:
# ── EXPORTAR ───────────────────────────────────────────
df_final.to_csv(SALIDA, index=False, encoding="utf-8-sig")